# Dictionary-Based Content Analysis and Its Limits

**Computational Text Analysis Pack · Module 15 — Rule-Based and Symbolic Methods**

*Technique:* General Inquirer-style category counting; why sentiment lexicons fail here

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/04-rule-based-methods/15d_dictionary_content_analysis.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

The oldest computational method in the humanities that is still in daily use:
count how often a text uses words from a predefined category list. The **General
Inquirer** (Stone et al., 1966) is the ancestor and is still distributed — 11,789
words across 182 categories — and every sentiment tool you have heard of is the
same idea with a different word list.

It is transparent, reproducible, and requires no training data. It is also, on
the corpora this course cares about, frequently wrong in ways worth measuring.

## The tooling

**General Inquirer** (Harvard IV-4 + Lasswell), **LIWC** (psycholinguistic
categories, commercial), the **NRC Emotion Lexicon** (Mohammad),
**SentiWordNet** (built over WordNet synsets), and **VADER** (Hutto & Gilbert,
rule-based, tuned for social media — it combines a lexicon with grammatical
heuristics for negation and intensifiers).

NLTK ships VADER and a WordNet interface. The documented limitations are
consistent across all of them: restricted vocabulary, poor handling of sarcasm,
negation and domain jargon, and — the one that matters most here — they are
English lexicons.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# A General Inquirer-style dictionary, in the format those dictionaries use:
# a category name and a word list. Small, but the mechanism is identical.
CATEGORIES = {
    "AUTHORITY": {"king", "chief", "governor", "command", "rule", "law",
                  "order", "власть", "authority", "empire", "sovereign",
                  "obey", "subject", "govern", "throne"},
    "TRADE":     {"trade", "goods", "market", "merchant", "sell", "buy",
                  "price", "profit", "cargo", "gold", "ivory", "cotton"},
    "RELIGION":  {"god", "church", "mission", "christian", "prayer", "faith",
                  "heathen", "soul", "priest", "gospel", "sacred"},
    "VIOLENCE":  {"war", "battle", "kill", "attack", "blood", "weapon",
                  "slave", "captive", "fight", "destroy", "punish"},
    "NATURE":    {"river", "forest", "mountain", "rain", "sun", "tree",
                  "animal", "land", "sea", "climate", "soil"},
}
lookup = {w: cat for cat, words in CATEGORIES.items() for w in words}
print(f"{len(CATEGORIES)} categories, {len(lookup)} entries")

In [ ]:
def score(text):
    toks = tokenise(text)
    counts = Counter(lookup[w] for w in toks if w in lookup)
    n = len(toks)
    return {cat: counts.get(cat, 0) / n * 1000 for cat in CATEGORIES}

works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
scored = pd.DataFrame([score(t) for t in works["text"]])
scored["author"] = works["author"].values
by_author = (scored.groupby("author").mean(numeric_only=True)
             .loc[works["author"].value_counts().head(8).index])
print("category rate per 1,000 words, by author")
print(by_author.to_string(float_format=lambda v: f"{v:.2f}"))

In [ ]:
ax = by_author.plot(kind="barh", stacked=True, figsize=(9, 4.5))
ax.set_xlabel("occurrences per 1,000 words")
ax.set_title("General Inquirer-style category profile")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()

## The three failures, demonstrated

Dictionary methods fail in specific, documented ways. Each is easy to show and
easy to forget when the output is a tidy table.

In [ ]:
# Failure 1: negation. The lexicon cannot see it.
pairs = [("The mission was a great success.", "The mission was no success at all."),
         ("They were sacred to the people.", "Nothing was sacred to them."),
         ("He obeyed the king.", "He refused to obey the king.")]
print("Failure 1 — NEGATION\n")
for pos, neg in pairs:
    print(f"  '{pos}'\n     -> {  {k: v for k, v in score(pos).items() if v} }")
    print(f"  '{neg}'\n     -> { {k: v for k, v in score(neg).items() if v} }")
    print()
print("Identical or near-identical scores. VADER handles simple negation with")
print("grammatical heuristics; a bare word list cannot handle it at all.")

In [ ]:
# Failure 2: the words are English.
print("Failure 2 — LANGUAGE\n")
rows = []
for lang, g in df.groupby("language"):
    joined = " ".join(g["text"].head(50))
    toks = tokenise(joined)
    hit = sum(1 for w in toks if w in lookup)
    rows.append({"language": lang, "tokens": len(toks),
                 "dictionary_hits": hit, "coverage": hit / max(len(toks), 1)})
cov = pd.DataFrame(rows).sort_values("coverage", ascending=False)
print(cov.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("Coverage collapses outside English. Running an English sentiment lexicon")
print("over Yoruba or Hausa does not give you a weak signal — it gives you a")
print("measurement of how many English loanwords the text contains.")

In [ ]:
# Failure 3: the category list is an argument, not an observation.
print("Failure 3 — THE DICTIONARY IS A CLAIM\n")
overlap = Counter()
for w, cat in lookup.items():
    overlap[w] += 1
print("Consider where 'slave' sits: this dictionary files it under VIOLENCE.")
print("File it under TRADE instead and every result above changes, because in")
print("this corpus the word appears in commercial contexts as often as violent")
print("ones.\n")
for cat in ["VIOLENCE", "TRADE"]:
    test = {c: set(w) for c, w in CATEGORIES.items()}
    test["VIOLENCE"] = test["VIOLENCE"] - {"slave"}
    test["TRADE"] = test["TRADE"] | ({"slave"} if cat == "TRADE" else set())
    lk = {w: c for c, ws in test.items() for w in ws}
    toks = tokenise(" ".join(works["text"]))
    counts = Counter(lk[w] for w in toks if w in lk)
    total = sum(counts.values())
    print(f"  with 'slave' in {cat:<9} VIOLENCE={counts['VIOLENCE']/total:.1%}"
          f"  TRADE={counts['TRADE']/total:.1%}")
print()
print("The categorisation decision produced the finding. That is not a bug to")
print("be fixed — it is why the dictionary must be published alongside results.")

### 🔧 Your turn 1

Build a category for a theme you actually study, run it over the corpus, and then ask a colleague to build the same category independently. How much do your word lists overlap?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Independently built lists for the same concept typically overlap around half,
which is the uncomfortable finding at the heart of dictionary methods and the
reason module 17 covers inter-annotator agreement. The response is not to
abandon the method — it is transparent and reproducible in a way most
alternatives are not — but to treat the dictionary as a research object: publish
it, report who built it, and where possible report agreement between independent
versions. That is what the content-analysis tradition has done since 1966.

</details>